# Côte d'Ivoire - PROMIRE project / UN-REDD

2 · Area estimates and uncertainties, national and by phytogeographic zone - SBAE stratified estimator (STR), 2021-2025
-------
For each activity class (SNF, SF, SDegF, Deg, Def, Gain), this notebook computes the estimated area and its uncertainty (standard error, uncertainty and 90% confidence interval) at the national level and for each of the four phytogeographic zones, then compares the sum of the zones with the national estimate. It follows the stratified estimator (STR) of Cochran (1977) / Olofsson et al. (2014), used in all the FAO SBAE tools.

**Inputs**: `results/CIV_activity_samples_2021_2025_en.csv` (produced by notebook `1`) and the phytogeographic zones (`data/ZnePhytoGeo_CI.shp`).

**Outputs** (`results/` folder), dynamic Excel workbooks produced by the module `sbae_excel_en.py`:

- `CIV_area_estimates_STR_2021_2025_by_zone_en.xlsx`: national and zone estimates;
- `CIV_area_estimates_STR_2021_2025_by_zone_ERP_en.xlsx`: the same estimates for the territory of the ERP (Emission Reductions Program, *PRE* in French) only, last section.

*English version of `2_CIV_estimation_superficies_STR_2021_2025.ipynb`: same calculations, same results.*

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

In [ ]:
# Samples located outside the phytogeographic zones (along the national border):
# True = they are assigned to the nearest zone; False = they are excluded
assign_nearest_zone = True

## Loading the phytogeographic zones

In [ ]:
zones = gpd.read_file('data/ZnePhytoGeo_CI.shp')
zones.plot()

In [ ]:
print(zones.Zone)

## Loading the data

In [ ]:
data_path = 'results/CIV_activity_samples_2021_2025_en.csv'   # output of notebook 1
df = pd.read_csv(data_path)
len(df)

In [ ]:
df.head()

### Data quality control

In [ ]:
duplicated = df['plotid'].duplicated().any()
if duplicated:
    print("Problem: duplicated plotid")
else:
    print("All correct: the plotid are unique")

In [ ]:
missing_strata = df['strata'].isna().sum()
print(f"{missing_strata} samples have no stratum (outside the territory or masked area) and are excluded from the estimator")

df_valid = df.dropna(subset=['strata']).copy()
df_valid['strata'] = df_valid['strata'].astype(int)
len(df_valid)

In [ ]:
count_values = df_valid['activity'].value_counts()
print(count_values)

## Assigning a phytogeographic zone to each sample

In [ ]:
gdf = gpd.GeoDataFrame(df,geometry=gpd.points_from_xy(df.center_lon, df.center_lat),crs="EPSG:4326").to_crs('EPSG:32630')
gdf_w_zones = gpd.sjoin_nearest(gdf[['plotid','geometry']],zones,how='left', distance_col='zdistance')

In [ ]:
gdf_w_zones.zdistance.plot.hist()

In [ ]:
print("Number of samples outside the zones:", len(gdf_w_zones[gdf_w_zones.zdistance > 0]))
df_valid = df_valid.merge(gdf_w_zones[['plotid', 'Zone', 'zdistance']]
                          .rename(columns={'Zone': 'zone'}), on='plotid', how='left')
if assign_nearest_zone:
    print('These samples are assigned to the nearest zone')
else:
    df_valid = df_valid[df_valid.zdistance == 0]
    print('These samples are excluded')

In [ ]:
print("Number of samples with a stratum:", len(df_valid))
df_valid['zone'].value_counts()

In [ ]:
df_valid['activity'].value_counts()

## Quick look at the data

### Number of samples by activity class

In [ ]:
count_values = df_valid.groupby('zone')['activity'].value_counts().unstack('zone')
print(count_values)

### Excluding the samples with activity ND

Notebook `1` assigns the activity `ND` to the samples whose level-1 class is missing (or `no_data`) in 2021 or in 2025. They are excluded from the estimator.

In [ ]:
df_valid = df_valid[df_valid.activity!='ND']
len(df_valid)

### Number of samples by activity class, zone and stratum

In [ ]:
count_values = (
    df_valid.groupby(['zone', 'strata'])['activity'].value_counts()
    .unstack(['zone', 'strata'], fill_value=0)
    .sort_index(axis=1)
)
print(count_values)

In [ ]:
count_values = (
    df_valid.groupby(['zone', 'strata'])['activity'].value_counts(normalize=True)
    .unstack(['zone', 'strata'], fill_value=0)
    .sort_index(axis=1) * 100
)
print(count_values)

The following table gives $\hat{p}_{hk}\,(1-\hat{p}_{hk})$ for each class and each zone × stratum combination: it is the variance term that enters the standard error of the estimator (see the equations below). It is largest for classes close to 50% and very small for rare classes.

In [ ]:
(count_values/100) * (1 - (count_values/100))

## Stratified area estimator (STR) — national level

For a stratified sampling design with $H$ strata, the area proportion of class $k$ is the mean of the sample proportions of each stratum, weighted by the stratum areas:

$$
\hat{p}_k \;=\; \sum_{h=1}^{H} W_h\,\hat{p}_{hk},
\qquad
\hat{p}_{hk} \;=\; \frac{n_{hk}}{n_h},
\qquad
W_h \;=\; \frac{A_h}{A}
$$

$$
\widehat{\mathrm{SE}}(\hat{p}_k) \;=\; \sqrt{\;\sum_{h=1}^{H} W_h^{2}\;\frac{\hat{p}_{hk}\,\left(1-\hat{p}_{hk}\right)}{n_h-1}\;}
$$

The area, its standard error, the uncertainty $U$ (at 90%) and the confidence interval follow:

$$
\hat{A}_k \;=\; A\,\hat{p}_k,
\qquad
\widehat{\mathrm{SE}}(\hat{A}_k) \;=\; A\;\widehat{\mathrm{SE}}(\hat{p}_k),
\qquad
U(\hat{A}_k) \;=\; 1.645\;\widehat{\mathrm{SE}}(\hat{A}_k),
\qquad
\mathrm{CI}_{90\%}(\hat{A}_k) \;=\; \hat{A}_k \,\pm\, U(\hat{A}_k)
$$

| Symbol | Meaning |
|:--|:--|
| $A_h$ | mapped area of stratum $h$ |
| $A = \sum_h A_h$ | total mapped area |
| $W_h$ | area weight of stratum $h$ |
| $n_h$ | number of samples in stratum $h$ |
| $n_{hk}$ | number of samples of stratum $h$ assigned to class $k$ by the reference data (CEO) |
| $U$ | uncertainty: $z \cdot \widehat{\mathrm{SE}}$, with $z = 1.645$ for 90% |

Reference: Olofsson, P. et al. (2014) *Good practices for estimating area and assessing accuracy of land change*, Remote Sensing of Environment 148.

### Estimation by zone — zone × stratum post-stratification

The phytogeographic zones are not strata of the sampling design: they cut across the strata. The mapped area of each stratum $h$ in each zone $z$, $A_{hz}$, is known, and the samples of stratum $h$ that fall in zone $z$ are a random sample of cell $(h, z)$. Each zone is therefore treated as a stratified population in its own right, with its own weights:

$$
W_{hz} \;=\; \frac{A_{hz}}{A_z},
\qquad
A_z \;=\; \sum_{h=1}^{H} A_{hz}
$$

$$
\hat{p}_{kz} \;=\; \sum_{h=1}^{H} W_{hz}\,\hat{p}_{hkz},
\qquad
\hat{p}_{hkz} \;=\; \frac{n_{hkz}}{n_{hz}}
$$

$$
\widehat{\mathrm{SE}}(\hat{p}_{kz}) \;=\; \sqrt{\;\sum_{h=1}^{H} W_{hz}^{2}\;\frac{\hat{p}_{hkz}\,\left(1-\hat{p}_{hkz}\right)}{n_{hz}-1}\;}
$$

$$
\hat{A}_{kz} \;=\; A_z\,\hat{p}_{kz},
\qquad
\widehat{\mathrm{SE}}(\hat{A}_{kz}) \;=\; A_z\;\widehat{\mathrm{SE}}(\hat{p}_{kz}),
\qquad
\sum_k \hat{A}_{kz} \;=\; A_z
$$

| Symbol | Meaning |
|:--|:--|
| $A_{hz}$ | mapped area of stratum $h$ in zone $z$ |
| $A_z$ | mapped area of zone $z$ |
| $n_{hz}$ | number of samples of stratum $h$ located in zone $z$ |
| $n_{hkz}$ | number of these samples assigned to class $k$ |

The sum of the classes of a zone equals its mapped area, and the variance due to the random number of samples falling in each zone is removed.

### Stratum areas (weights Wh)

In [ ]:
strata_sizes = (
    df_valid['strata'].value_counts()
    .rename_axis('stratum')
    .reset_index(name='n_h')
)
# Mapped area of each stratum (ha), assigned by stratum number
strata_sizes['area_ha'] = strata_sizes['stratum'].map({1: 19080547.4, 2: 10923787.62, 3: 2037564.502})
strata_sizes

In [ ]:
total_area_ha = strata_sizes['area_ha'].sum()
print('Total area (ha):',total_area_ha)
Wh = strata_sizes.set_index('stratum')['area_ha'] / total_area_ha
Wh

In [ ]:
print("Number of samples by zone and stratum:")
pd.crosstab(df_valid['zone'], df_valid['strata'], margins=True)

### Stratum areas by zone (weights W<sub>hz</sub>)

In [ ]:
# Mapped area of each stratum in each zone (km², converted to ha)
# GEE script: https://code.earthengine.google.com/6d246c730ae5efc836ced21ef90d74ac


area_hz_km2 = pd.DataFrame({
    'Secteur Ombrophile':    [28127.4082,	21377.56608,	8586.479557],
    'Secteur Soudanais':     [86493.21318,	39087.48208,	2729.667459],
    'Secteur Sub Soudanais': [21759.34011,	13883.2849,	1990.485385],
    'Secteur mésophile':     [54425.51251,	34889.54318,	7069.012614],
}, index=pd.Index([1, 2, 3], name='stratum')).T
area_hz_km2.index.name = 'zone'
area_hz = area_hz_km2 * 100
area_hz['total'] = area_hz.sum(axis=1)
area_hz

In [ ]:
# Check: sum of the zones by stratum vs national stratum areas
chk_strata = pd.DataFrame({
    'sum_zones_ha': area_hz[[1, 2, 3]].sum(),
    'national_ha': strata_sizes.set_index('stratum')['area_ha'],
})
chk_strata['diff_pct'] = (chk_strata['sum_zones_ha'] / chk_strata['national_ha'] - 1) * 100
print(chk_strata, '\n')
print(f"Total area of the zones: {area_hz['total'].sum():,.1f} ha vs {total_area_ha:,.1f} ha at the national level")

Wh_zone = area_hz[[1, 2, 3]].div(area_hz['total'], axis=0)
Wh_zone

### Stratified estimates by activity class and by zone

In [ ]:
def stratified_area_estimate(df, class_col, stratum_col, weights, total_area_ha, z=1.645, verbose=False):
    """STR area estimator (stratified random sampling) -- Cochran (1977) / Olofsson et al. (2014).
    `weights` is a pandas Series indexed by stratum value, containing Wh = area_h / total_area."""
    n_h = df.groupby(stratum_col).size()
    rows = []
    for cls in sorted(df[class_col].dropna().unique()):
        n_hk = (
            df[df[class_col] == cls]
            .groupby(stratum_col).size()
            .reindex(n_h.index, fill_value=0)
        )
        p_h = n_hk / n_h
        w_h = weights.reindex(n_h.index)
        p_hat = (w_h * p_h).sum()
        var_terms = w_h**2 * p_h * (1 - p_h) / (n_h - 1)
        var_hat = var_terms.replace([np.inf, -np.inf], np.nan).sum()
        se_hat = np.sqrt(var_hat)
        area = p_hat * total_area_ha
        se_area = se_hat * total_area_ha
        if verbose:
            print (cls, var_terms * area)
        rows.append({
            'activity': cls,
            'n_sample': int((df[class_col] == cls).sum()),
            'proportion': p_hat,
            'area_ha': area,
            'se_ha': se_area,
            'ci90_ha': z * se_area,
            'ci90_pct_of_area': (z * se_area / area * 100) if area else np.nan,
        })

    return pd.DataFrame(rows).sort_values('area_ha', ascending=False).reset_index(drop=True)

results = stratified_area_estimate(df_valid, 'activity', 'strata', Wh, total_area_ha, verbose=False)
results

In [ ]:
def stratified_area_estimate_by_zone(df, class_col, stratum_col, zone_col, area_hz, z=1.645):
    """Zone x stratum post-stratification: the STR estimator is applied separately to each zone,
    with its own weights W_hz = A_hz / A_z and the samples of the zone only.
    `area_hz` is a DataFrame indexed by zone, with one column per stratum (areas in ha)."""
    out = []
    for zn in sorted(df[zone_col].dropna().unique()):
        a_h = area_hz.loc[zn, [1, 2, 3]]
        res = stratified_area_estimate(df[df[zone_col] == zn], class_col, stratum_col, a_h / a_h.sum(), a_h.sum(), z=z)
        res.insert(0, 'zone', zn)
        out.append(res)
    return pd.concat(out, ignore_index=True)


results_zone = stratified_area_estimate_by_zone(df_valid, 'activity', 'strata', 'zone', area_hz)
results_zone['pct_of_zone'] = results_zone['area_ha'] / results_zone.groupby('zone')['area_ha'].transform('sum') * 100
results_zone

In [ ]:
print("Estimated area by class and by zone (ha):")
results_zone.pivot(index='activity', columns='zone', values='area_ha').fillna(0)

In [ ]:
print("Relative uncertainty at 90% (in % of the estimated area):")
results_zone.pivot(index='activity', columns='zone', values='ci90_pct_of_area')

### Consistency checks

In [ ]:
# Consistency checks
# 1. With post-stratification, the sum of the classes of a zone equals its mapped area
print((results_zone.groupby('zone')['area_ha'].sum() - area_hz['total']).rename('diff_ha').to_string(), '\n')

# 2. Zone/class combinations with few samples
low_sample = results_zone[results_zone['n_sample'] < 10]
if len(low_sample):
    print("Zone/class combinations with fewer than 10 samples -- interpret their SE/uncertainty with caution:")
    print(low_sample[['zone', 'activity', 'n_sample']].to_string(index=False))

### Area estimates by zone with their 90% uncertainty

In [ ]:
zone_list = sorted(results_zone['zone'].unique())
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
for ax, zn in zip(axes.flat, zone_list):
    order = results_zone[results_zone['zone'] == zn].sort_values('area_ha', ascending=True)
    ax.barh(order['activity'], order['area_ha'], xerr=order['ci90_ha'], capsize=4, color='#4C72B0')
    ax.set_title(zn)
    ax.set_xlabel('Area (ha)')
fig.suptitle("Stratified area estimates by activity class and by zone (± 90% uncertainty)")
plt.tight_layout()
plt.show()

In [ ]:
# Change classes only (without SNF/SF), for readability
chg = results_zone[~results_zone['activity'].isin(['SNF', 'SF'])]
area_p = chg.pivot(index='activity', columns='zone', values='area_ha')
ci_p = chg.pivot(index='activity', columns='zone', values='ci90_ha')
ax = area_p.plot.bar(yerr=ci_p, capsize=3, figsize=(11, 5))
ax.set_ylabel('Area (ha)')
ax.set_title('Area of the change classes by zone (± 90% uncertainty)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Comparison: national SBAE vs SBAE by zone

The zones are estimated from disjoint sets of samples: the zone estimates add up, and so do their variances. For each class $k$:

$$
\hat{A}_k^{\,\Sigma z} \;=\; \sum_{z} \hat{A}_{kz},
\qquad
\widehat{\mathrm{SE}}\big(\hat{A}_k^{\,\Sigma z}\big) \;=\; \sqrt{\;\sum_{z} \widehat{\mathrm{SE}}(\hat{A}_{kz})^{2}\;}
$$

This sum of the zones is compared with the national estimate $\hat{A}_k$:

$$
\Delta_k \;=\; \hat{A}_k^{\,\Sigma z} - \hat{A}_k,
\qquad
\Delta_k\,(\%) \;=\; 100\,\frac{\Delta_k}{\hat{A}_k},
\qquad
\rho_k \;=\; \frac{\widehat{\mathrm{SE}}\big(\hat{A}_k^{\,\Sigma z}\big)}{\widehat{\mathrm{SE}}(\hat{A}_k)}
$$

The two estimates do not coincide exactly: the estimators differ (zone × stratum post-strata vs strata only) and the total area of the zones $\sum_z A_z$ is slightly smaller than the national stratum area $A$ (see the area check above). $\rho_k < 1$ indicates that post-stratification by zone reduces the uncertainty.

In [ ]:
Z90 = 1.645

# Aggregation of the SBAE by zone: sum of the areas, sum of the variances
agg_zones = (results_zone.assign(var_ha=results_zone['se_ha'] ** 2)
             .groupby('activity')
             .agg(area_ha=('area_ha', 'sum'), var_ha=('var_ha', 'sum')))
agg_zones['se_ha'] = np.sqrt(agg_zones.pop('var_ha'))

cmp_nat = (results.set_index('activity')[['n_sample', 'area_ha', 'se_ha']]
           .join(agg_zones, lsuffix='_nat', rsuffix='_zones'))
for s in ['nat', 'zones']:
    cmp_nat[f'ci90_ha_{s}'] = Z90 * cmp_nat[f'se_ha_{s}']
    cmp_nat[f'ci90_pct_{s}'] = cmp_nat[f'ci90_ha_{s}'] / cmp_nat[f'area_ha_{s}'] * 100
cmp_nat['diff_ha'] = cmp_nat['area_ha_zones'] - cmp_nat['area_ha_nat']
cmp_nat['diff_pct'] = cmp_nat['diff_ha'] / cmp_nat['area_ha_nat'] * 100
cmp_nat['ratio_se'] = cmp_nat['se_ha_zones'] / cmp_nat['se_ha_nat']
cmp_nat = cmp_nat[['n_sample',
                   'area_ha_nat', 'se_ha_nat', 'ci90_ha_nat', 'ci90_pct_nat',
                   'area_ha_zones', 'se_ha_zones', 'ci90_ha_zones', 'ci90_pct_zones',
                   'diff_ha', 'diff_pct', 'ratio_se']].sort_values('area_ha_nat', ascending=False)

print(f"Total area: national {total_area_ha:,.1f} ha; sum of the zones {area_hz['total'].sum():,.1f} ha "
      f"({(area_hz['total'].sum() / total_area_ha - 1) * 100:+.3f} %)")
cmp_nat

In [ ]:
# Change classes: national SBAE vs sum of the SBAE by zone (± 90% uncertainty)
chg_cmp = cmp_nat.drop(index=['SNF', 'SF'], errors='ignore')
x = np.arange(len(chg_cmp))
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - 0.2, chg_cmp['area_ha_nat'], 0.4, yerr=chg_cmp['ci90_ha_nat'], capsize=4, label='National SBAE', color='#4C72B0')
ax.bar(x + 0.2, chg_cmp['area_ha_zones'], 0.4, yerr=chg_cmp['ci90_ha_zones'], capsize=4, label='Sum of the SBAE by zone', color='#55A868')
ax.set_xticks(x, chg_cmp.index)
ax.set_ylabel('Area (ha)')
ax.set_title('Change classes: national SBAE vs sum of the SBAE by zone (± 90% uncertainty)')
ax.legend()
plt.tight_layout()
plt.show()

## Export to a dynamic Excel workbook

The workbook is produced by the module `sbae_excel_en.py` (same folder as the notebooks). It is **self-contained and fully formula-driven**: the only typed-in values are the inputs (yellow cells):

- sheet *Samples*: stratum, zone and class of each sample → counts $n_{hk}$ and $n_{hkz}$ with `COUNTIFS`;
- sheet *Inputs*: confidence level, national stratum areas $A_h$ and areas by zone $A_{hz}$.

All the other cells (weights, proportions, areas, standard errors, uncertainties, national vs zones comparison, charts) are formulas: changing an input recalculates the whole workbook, which makes it possible to test scenarios.

In [ ]:
import os
from sbae_excel_en import build_workbook

xlsx_path = 'results/CIV_area_estimates_STR_2021_2025_by_zone_en.xlsx'
os.makedirs(os.path.dirname(xlsx_path), exist_ok=True)
xl = build_workbook(
    xlsx_path,
    plots=df_valid,
    area_hz=area_hz[[1, 2, 3]],
    area_h=strata_sizes.set_index('stratum')['area_ha'],
    confidence=0.90,
    data_source=os.path.basename(data_path),
    zone_rule='assigned to the nearest zone' if assign_nearest_zone else 'excluded',
    area_source='GEE script https://code.earthengine.google.com/4b8e3c9068b9311da3602896b234c14c',
)
print('Excel workbook written:', os.path.abspath(xlsx_path))

In [ ]:
# Check: the workbook values (same equations, computed by sbae_excel_en) are identical to those of the notebook
chk_zone = xl['zone'].merge(results_zone, on=['zone', 'activity'], how='left', suffixes=('_xl', '')).fillna(0)
chk_nat = xl['national'].merge(results, on='activity', suffixes=('_xl', ''))
chk_sum = xl['sum_zones'].set_index('activity').join(cmp_nat[['area_ha_zones', 'se_ha_zones']])
for name, a, b in [('zones: area', chk_zone['area_ha_xl'], chk_zone['area_ha']),
                   ('zones: standard error', chk_zone['se_ha_xl'], chk_zone['se_ha']),
                   ('national: area', chk_nat['area_ha_xl'], chk_nat['area_ha']),
                   ('national: standard error', chk_nat['se_ha_xl'], chk_nat['se_ha']),
                   ('sum of the zones: area', chk_sum['area_ha'], chk_sum['area_ha_zones']),
                   ('sum of the zones: standard error', chk_sum['se_ha'], chk_sum['se_ha_zones'])]:
    print(f"{name:<36} max difference = {np.max(np.abs(a - b)):.2e} ha  ->  {'OK' if np.allclose(a, b) else 'MISMATCH'}")

## Estimates for the ERP

The calculation is repeated for the samples located in the ERP only (column `pre` produced by notebook `1`), with the mapped stratum areas **within the ERP**, by zone ($A_{hz}$ of the ERP):

- **ERP SBAE**: STR estimator with the stratum areas of the ERP, $A_h = \sum_z A_{hz}$;
- **SBAE by zone within the ERP**: zone × stratum post-stratification, as above, with the $A_{hz}$ of the ERP.

The ERP does not overlap the Secteur Soudanais. A zone of the ERP without any sample cannot be estimated: it is removed from the estimation by zone (its area is still counted in the ERP SBAE, and the difference appears in the area check of the workbook).

In [ ]:
# Mapped area of each stratum within the ERP, by zone (km², converted to ha)
area_hz_erp_km2 = pd.DataFrame({
    'Secteur Ombrophile':    [17262.20123016427,  12174.209957087007, 4988.553467034733],
    'Secteur Sub Soudanais': [24.385942521542056, 9.404869314886117,  0.24926422661324296],
    'Secteur mésophile':     [6623.2923265831605, 4158.777891905002,  710.699415400892],
}, index=pd.Index([1, 2, 3], name='stratum')).T
area_hz_erp_km2.index.name = 'zone'
area_hz_erp = area_hz_erp_km2 * 100
area_hz_erp['total'] = area_hz_erp.sum(axis=1)
area_h_erp = area_hz_erp[[1, 2, 3]].sum()                     # stratum areas within the ERP (ha)
area_hz_erp

In [ ]:
df_erp = df_valid[df_valid['pre']].copy()
assert 'Secteur Soudanais' not in set(df_erp['zone']), 'ERP sample assigned to the Secteur Soudanais'
print(len(df_erp), 'valid samples in the ERP')
print("Number of samples by zone and stratum in the ERP:")
pd.crosstab(df_erp['zone'], df_erp['strata'], margins=True)

In [ ]:
# ERP zones without any sample: removed from the estimation by zone
zones_without_samples = [zn for zn in area_hz_erp.index if zn not in set(df_erp['zone'])]
for zn in zones_without_samples:
    print(f"{zn}: no sample for {area_hz_erp.loc[zn, 'total']:,.0f} ha "
          f"({area_hz_erp.loc[zn, 'total'] / area_hz_erp['total'].sum():.2%} of the ERP) -> not estimated by zone")
area_hz_erp_est = area_hz_erp.drop(index=zones_without_samples)

In [ ]:
# ERP SBAE (ERP strata) and SBAE by zone within the ERP
total_area_erp_ha = area_h_erp.sum()
results_erp = stratified_area_estimate(df_erp, 'activity', 'strata', area_h_erp / total_area_erp_ha, total_area_erp_ha)
results_zone_erp = stratified_area_estimate_by_zone(df_erp, 'activity', 'strata', 'zone', area_hz_erp_est)
results_zone_erp['pct_of_zone'] = results_zone_erp['area_ha'] / results_zone_erp.groupby('zone')['area_ha'].transform('sum') * 100
print(f'ERP area: {total_area_erp_ha:,.1f} ha')
results_erp

In [ ]:
print("ERP: estimated area by class and by zone (ha):")
results_zone_erp.pivot(index='activity', columns='zone', values='area_ha').fillna(0)

In [ ]:
print("ERP: relative uncertainty at 90% (in % of the estimated area):")
results_zone_erp.pivot(index='activity', columns='zone', values='ci90_pct_of_area')

In [ ]:
low_sample = results_zone_erp[results_zone_erp['n_sample'] < 10]
if len(low_sample):
    print("ERP: zone/class combinations with fewer than 10 samples -- interpret their SE/uncertainty with caution:")
    print(low_sample[['zone', 'activity', 'n_sample']].to_string(index=False))

In [ ]:
xlsx_path_erp = 'results/CIV_area_estimates_STR_2021_2025_by_zone_ERP_en.xlsx'
xl_erp = build_workbook(
    xlsx_path_erp,
    plots=df_erp,
    area_hz=area_hz_erp_est[[1, 2, 3]],
    area_h=area_h_erp,
    confidence=0.90,
    title="Côte d'Ivoire — ERP — SBAE by phytogeographic zone",
    territory='ERP',
    data_source=os.path.basename(data_path) + ' (samples located in the ERP)',
    zone_rule='assigned to the nearest zone' if assign_nearest_zone else 'excluded',
    area_source='stratum areas within the ERP, by zone (provided)'
                + (f" ; zone without samples not estimated: {', '.join(zones_without_samples)}" if zones_without_samples else ''),
)
print('Excel workbook written:', os.path.abspath(xlsx_path_erp))

In [ ]:
# Check: the values of the ERP workbook are identical to those of the notebook
chk_zone = xl_erp['zone'].merge(results_zone_erp, on=['zone', 'activity'], how='left', suffixes=('_xl', '')).fillna(0)
chk_erp = xl_erp['national'].merge(results_erp, on='activity', suffixes=('_xl', ''))
for name, a, b in [('zones: area', chk_zone['area_ha_xl'], chk_zone['area_ha']),
                   ('zones: standard error', chk_zone['se_ha_xl'], chk_zone['se_ha']),
                   ('ERP: area', chk_erp['area_ha_xl'], chk_erp['area_ha']),
                   ('ERP: standard error', chk_erp['se_ha_xl'], chk_erp['se_ha'])]:
    print(f"{name:<36} max difference = {np.max(np.abs(a - b)):.2e} ha  ->  {'OK' if np.allclose(a, b) else 'MISMATCH'}")